In [ ]:
### Author: Demesvar D. Destin, DrPH
### Year: 2026
### Contact: demesvardestin@gmail.com

In [ ]:
#### BEGIN GENETIC ANALYSES ####

print("BEGINNING GENETIC ANALYSES!")

### BEGING LIBRARY IMPORT ###

import sys
import os
import subprocess
import zoneinfo
import pandas as pd
import numpy as np
import math
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
from datetime import datetime, timedelta
from scipy import stats
from scipy.stats import chi2_contingency
from scipy.stats import chisquare
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.model_selection import train_test_split

### END LIBRARY IMPORT ###

### BEGIN DATAFRAME DOWNLOAD & SETUP ###
print("Downloading dataframes from AoU...")

## NOTE: In AoU Researcher Workbench, dataframes (dfs) are downloaded directly using SQL,
## but these are not included here due to data privacy rules. all dfs named below refer to
## the downloaded data that stayed in the Workbench.
## person_df: demographics
## survey_df: alcohol, smoking
## measurement_df: anthropometrics
## drug_df: medications
## condition_df: joint pain status

# drop duplicate rows in condition dataframe. one row signifies presence of at least one type of joint pain
condition_df.drop_duplicates(subset=['person_id'], inplace=True)

# rename 'standard_concept_name' column in condition_df
condition_df.rename(columns={'standard_concept_name': 'joint_pain'}, inplace=True)

print("Download completed. Merging dataframes...")

person_df.insert(loc=4, column='age', value=0)
measurement_df.insert(loc=3, column='height', value=0)
measurement_df.insert(loc=4, column='hip_circumference', value=0)
measurement_df.insert(loc=5, column='waist_circumference', value=0)
measurement_df.insert(loc=6, column='bri', value=0)
measurement_df.insert(loc=7, column='bmi', value=0)

# function to calculate age at treatment initiation
# takes date of birth (born) and AIT start date
def calculate_age(born, ait_start_date):
    try:
        birthday = born.replace(year = int(ait_start_date.year))
    # raised when birth date is February 29
    # and the current year is not a leap year
    except ValueError:
        birthday = born.replace(year = int(ait_start_date.year),
                  month = born.month + 1, day = 1)
    if birthday > ait_start_date:
        return ait_start_date.year - born.year - 1
    else:
        return ait_start_date.year - born.year

# function to calculate body roundness index (bri)
# takes waist_circumference and height as arguments
def calculate_bri(waist_circumference_cm, height_cm):

    # validate args to ensure positive numbers
    if waist_circumference_cm <= 0 or height_cm <= 0:
        return 0
    try:
        # core ratio term
        first_term = (waist_circumference_cm / (2 * math.pi))
        second_term = (0.5 * height_cm)
        ratio_term = (first_term*first_term) / (second_term*second_term)

        # check if the value inside the square root is non-negative
        value_under_sqrt = 1 - ratio_term

        # apply bri formula
        bri = 364.2 - 365.5 * value_under_sqrt
        return bri

    except ZeroDivisionError:
        print("error: height cannot be zero")
        return 0
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return 0

# create empty anthropometric df to collapse duplicate
# measurement_df rows into unique rows
anthro_data = {'person_id': '', 'height': 0.0, 'waist_circumference': 0.0,
               'hip_circumference': 0.0, 'cars': 0.0, 'bmi': 0.0, 'bri': 0.0, 'whr': 0.0}
anthro_df = pd.DataFrame(anthro_data, index=[0])

# loop through measurement_df, transfer height, weight,
# waist circumference, hip_circumference data to anthro_df
# based on unique person_id
for index, row in measurement_df.iterrows():
    if len(anthro_df.loc[anthro_df['person_id'] == row['person_id']]) == 0:
        new_row_df = pd.DataFrame([{'person_id': row['person_id'], 'height': 0.0,
                                    'waist_circumference': 0.0, 'hip_circumference': 0.0,
                                    'bmi': 0.0, 'bri': 0.0, 'whr': 0.0, 'cars': 0.0}])
        anthro_df = pd.concat([anthro_df, new_row_df], ignore_index=True)

    if pd.isna(row['value_as_number']):
        measurement_df.loc[index, 'value_as_number'] = 0.0

    if 'height' in row['standard_concept_name'] and row['value_as_number'] != 0:
        anthro_df.loc[anthro_df['person_id'] == row['person_id'], 'height'] = row['value_as_number']

    if 'Waist' in row['standard_concept_name'] and row['value_as_number'] != 0:
        anthro_df.loc[anthro_df['person_id'] == row['person_id'], 'waist_circumference'] = row['value_as_number']

    if 'hip' in row['standard_concept_name'] and row['value_as_number'] != 0:
        anthro_df.loc[anthro_df['person_id'] == row['person_id'], 'hip_circumference'] = row['value_as_number']

    if 'BMI' in row['standard_concept_name'] and row['value_as_number'] != 0:
        anthro_df.loc[anthro_df['person_id'] == row['person_id'], 'bmi'] = row['value_as_number']

## drop anthro_df index 0 (0 values)
anthro_df.drop(0, inplace=True)

# drop measurement_df duplicates. this df is not used from here on out
measurement_df.drop_duplicates(subset=['person_id'], inplace=True)

# drop duplicate rows in condition dataframe. one row signifies presence of at least one type of joint pain
condition_df = condition_df.sort_values('condition_start_datetime', ascending=False)
condition_df.drop_duplicates(subset=['person_id'], inplace=True, keep='first')

# rename 'standard_concept_name' column in condition_df
condition_df.rename(columns={'standard_concept_name': 'joint_pain'}, inplace=True)
drug_df.rename(columns={'standard_concept_name': 'drugs'}, inplace=True)

drugs_to_keep = ['anastrozole', 'letrozole', 'exemestane', 'tamoxifen', 'paclitaxel', 'cabazitaxel', 'docetaxel']
pattern = '|'.join(drugs_to_keep)
drug_df = drug_df[drug_df['drugs'].str.contains(pattern, case=False, na=False)]

ait_to_keep = '|'.join(['anastrozole', 'letrozole', 'exemestane'])
others = '|'.join(['tamoxifen', 'paclitaxel', 'cabazitaxel', 'docetaxel'])

ait_df = drug_df[drug_df['drugs'].str.contains(ait_to_keep, case=False, na=False)]
other_drugs_df = drug_df[drug_df['drugs'].str.contains(others, case=False, na=False)]

ait_df = ait_df.sort_values('drug_exposure_start_datetime', ascending=False)
ait_df.drop_duplicates(subset=['person_id'], inplace=True, keep='first')

other_drugs_df.drop_duplicates(subset=['person_id'], inplace=True, keep='first')

medication_data = {'person_id': '', 'ait_type': None, 'ait_start': datetime.now(tz=None)}
medication_df = pd.DataFrame(medication_data, index=[0])

# populate medication history
for index, row in drug_df.iterrows():
    if len(medication_df.loc[medication_df['person_id'] == row['person_id']]) == 0:
        new_row_df = pd.DataFrame([{'person_id': row['person_id'], 'prior_tamoxifen': 0,
                                    'pain_med': 0, 'prior_taxanes': 0, 'ait_start': datetime.now(tz=None)}])
        medication_df = pd.concat([medication_df, new_row_df], ignore_index=True)

for index, row in ait_df.iterrows():
    medication_df.loc[medication_df['person_id'] == row['person_id'], 'ait_start'] = pd.to_datetime(row['drug_exposure_start_datetime'])
    if 'anastrozole' in row['drugs']:
        medication_df.loc[medication_df['person_id'] == row['person_id'], 'ait_type'] = 'anastrozole'
    if 'letrozole' in row['drugs']:
        medication_df.loc[medication_df['person_id'] == row['person_id'], 'ait_type'] = 'letrozole'
    if 'exemestane' in row['drugs']:
        medication_df.loc[medication_df['person_id'] == row['person_id'], 'ait_type'] = 'exemestane'

for index, row in other_drugs_df.iterrows():
    if 'tamoxifen' in row['drugs']:
        medication_df.loc[medication_df['person_id'] == row['person_id'], 'prior_tamoxifen'] = 1
    elif 'paclitaxel' in row['drugs'] or 'docetaxel' in row['drugs'] or 'cabazitaxel' in row['drugs']:
        medication_df.loc[medication_df['person_id'] == row['person_id'], 'prior_taxanes'] = 1

## drop medication_df index 0 (0 values)
medication_df.drop(0, inplace=True)

# merge all dFS
merged_1 = pd.merge(person_df, condition_df, how='left')

merged_2 = pd.merge(merged_1, anthro_df, how='left')

joint_df = pd.merge(merged_2, medication_df, how='left')

### END DATAFRAME DOWNLOAD & SETUP ###

### BEGIN IMPUTATIONS & CLINICAL VARIABLES SETUP ###
print("Starting multivariate imputations...")

joint_df = joint_df[joint_df['ait_start'].notna()]

# standardize datetimes and strip timezones across entire columns
joint_df['condition_start_datetime'] = pd.to_datetime(joint_df['condition_start_datetime'], errors='coerce').dt.tz_localize(None)
joint_df['ait_start'] = pd.to_datetime(joint_df['ait_start'], errors='coerce').dt.tz_localize(None)

# calculate difference in days
time_diff_days = (joint_df['condition_start_datetime'] - joint_df['ait_start']).dt.days

# define conditions for case:
# - joint pain diagnosis AFTER or ON the day of AIT start (>= 0 days) AND
# - joint pain diagnosis WITHIN 5 years (<= 1825 days)
case_conditions = (
    joint_df['joint_pain'].notna() &
    (time_diff_days >= 0) &
    (time_diff_days <= 1825)
)

# 2 if case, 1 otherwise control
joint_df['pain'] = np.where(case_conditions, 2, 1)

print("Updated Cohort Distribution:")
print(joint_df['pain'].value_counts().rename({1: 'Controls', 2: 'Cases'}))

cases_df = joint_df[joint_df['pain'] == 2].copy()

# calculate time to event as a timedelta, then extract number of days
cases_df['time_to_aimss_days'] = (cases_df['condition_start_datetime'] - cases_df['ait_start']).dt.days

# calculate median and IQR
median_days = cases_df['time_to_aimss_days'].median()
median_months = median_days / 30.44 # standard average days in a month

q1_days = cases_df['time_to_aimss_days'].quantile(0.25)
q3_days = cases_df['time_to_aimss_days'].quantile(0.75)

print(f"Total AIMSS Cases Analyzed: {len(cases_df)}")
print(f"Median Time to Event: {median_days:.1f} days (~{median_months:.1f} months)")
print(f"Interquartile Range (IQR): {q1_days:.1f} to {q3_days:.1f} days")

# add binary states to pain column
for index, row in joint_df.iterrows():
    if pd.isna(row['ait_start']):
        joint_df.loc[index, 'ait_start'] = pd.to_datetime(datetime.now(tz=None))

    if not pd.isna(row['ait_start']):
        joint_df.loc[index, 'age'] = float(calculate_age(row['date_of_birth'].replace(tzinfo=None), row['ait_start'].replace(tzinfo=None)))

# set erroneous/zero values to NaN so they can be imputed; define physiological limits
cols_to_clean = ['height', 'waist_circumference', 'hip_circumference', 'bmi'] # raw metrics

for col in cols_to_clean:
    # set zeros to NaN
    joint_df[col] = joint_df[col].replace(0, np.nan)

    # set extreme physiological outliers to NaN
    if col == 'bmi':
        joint_df.loc[(joint_df[col] == 0) | (joint_df[col] is None), col] = np.nan
    if col == 'waist_circumference':
        joint_df.loc[(joint_df[col] == 0) | (joint_df[col] is None), col] = np.nan
    if col == 'hip_circumference':
        joint_df.loc[(joint_df[col] == 0) | (joint_df[col] is None), col] = np.nan
    if col == 'waist_circumference':
        joint_df.loc[(joint_df[col] == 0) | (joint_df[col] is None), col] = np.nan

# select columns to use for imputation model
impute_cols = ['age', 'height', 'waist_circumference', 'hip_circumference', 'bmi', 'pain']

# add race dummies to helper columns for better prediction
race_cols = [c for c in joint_df.columns if 'race_' in c]
prediction_matrix = joint_df[impute_cols + race_cols].copy()

# initialize and run MICE (IterativeImputer)
mice_imputer = IterativeImputer(max_iter=10, random_state=42, sample_posterior=True)
imputed_data = mice_imputer.fit_transform(prediction_matrix)

# put imputed values back into main df
imputed_df = pd.DataFrame(imputed_data, columns=prediction_matrix.columns, index=joint_df.index)

## qc: clamping
print("Clamping imputed values to physiological bounds...")

# height
if 'height' in imputed_df.columns:
    imputed_df['height'] = imputed_df['height'].clip(lower=100, upper=250)

# waist circumference
if 'waist_circumference' in imputed_df.columns:
    imputed_df['waist_circumference'] = imputed_df['waist_circumference'].clip(lower=50, upper=200)

# hip circumference
if 'hip_circumference' in imputed_df.columns:
    imputed_df['hip_circumference'] = imputed_df['hip_circumference'].clip(lower=50, upper=250)

# bmi
if 'bmi' in imputed_df.columns:
    imputed_df['bmi'] = imputed_df['bmi'].clip(lower=10, upper=50)

# update main joint_df with the new clamped values
for col in impute_cols:
    joint_df[col] = imputed_df[col]

# calculate indices based on imputed raw data
print("Re-calculating BRI and WHR based on imputed anthropometrics...")

for index, row in joint_df.iterrows():
    # calculate BRI
    if row['waist_circumference'] > 0 and row['height'] > 0:
        joint_df.loc[index, 'bri'] = calculate_bri(row['waist_circumference'], row['height'])

    # calculate WHR
    if row['waist_circumference'] > 0 and row['hip_circumference'] > 0:
        joint_df.loc[index, 'whr'] = row['waist_circumference'] / row['hip_circumference']

print("ImputationS and variable setup complete.")

### END IMPUTATIONS & CLINICAL VARIABLES SETUP ###

### BEGIN PHENO AND COVARIATE FILES SETUP ###

# rename the unique identifier column to 'FID' and 'IID'
joint_df['FID'] = joint_df['person_id']
joint_df['IID'] = joint_df['person_id']

# Select and reorder the required columns
pheno_data = joint_df[['FID', 'IID', 'age', 'bri', 'pain']]
column_names = ['FID', 'IID', 'age', 'bri', 'pain']

# Save the df to a text file without index/header
pheno_data.to_csv('joint_pain.phe', sep='\t', index=False, header=True, columns=column_names)

print("Files created")

# load phenotype file
# assuming it has columns: FID, IID, race (and others)
input_file = "joint_pain.phe"
df = pd.read_csv(input_file, sep='\s+', dtype={'FID': str, 'IID': str})

# check what the race categories actually look like
print("Original Race Categories:")
print(joint_df['race'].value_counts())

# create dummy variables
race_dummies = pd.get_dummies(joint_df['race'], prefix='race', dtype=int)

# concatenate the dummies back to the original df
df_final = pd.concat([df, race_dummies], axis=1)

# clean up column names
df_final.columns = df_final.columns.str.replace(' ', '_').str.replace('/', '_')

# save new covariates
output_file = "covariates_with_race.txt"
df_final.to_csv(output_file, sep='\t', index=False)

print("\nNew columns created:")
print(race_dummies.columns.tolist())
print(f"Saved to {output_file}")

print("Phenotype and covariate files have been created.")

### END PHENO & COVARIATE FILES SETUP ###

### BEGIN PLINK PREP (Note: This section is auto-generated by the AoU Researcher Workbench) ###

get_ipython().run_line_magic('env', 'DATASET_15055355_VCF_DIR=gs://fc-secure-8c4daf76-dd56-433e-abda-6a83a68f1db7/genomic-extractions/e08add8a-1fef-48d9-aff0-8b58709a1280/vcfs')

# The extraction workflow outputs a manifest file upon completion.
manifest_file = os.environ['DATASET_15055355_VCF_DIR'] + '/manifest.txt'

assert subprocess.run(['gsutil', '-q', 'stat', manifest_file]).returncode == 0, (
  "!" * 100 + "\n\n" +
  "VCF extraction has not completed.\n" +
  "Please monitor the extraction sidepanel for completion before continuing.\n\n" +
  "!" * 100
)

print("VCF extraction has completed, continuing")

get_ipython().run_cell_magic('bash', '', '\n# Download VCFs\n\nmkdir dataset_15055355_vcfs\ngsutil -m cp ${DATASET_15055355_VCF_DIR}/* dataset_15055355_vcfs/\n')

# Create a single merged VCF file
# This can take a few hours for a dataset with hundreds of participants

get_ipython().system('bcftools concat -a dataset_15055355_vcfs/*.vcf.gz -o dataset_15055355_vcfs/dataset_15055355_merged.vcf.gz')

get_ipython().system('plink --vcf dataset_15055355_vcfs/dataset_15055355_merged.vcf.gz --make-bed --out dataset_15055355_plink')

### END PLINK PREP ###

### BEGIN PLINK DESCRIPTIVE ANALYSIS

# summary stats on missing rates
get_ipython().system('plink2 --bfile dataset_15055355_plink --missing --allow-no-sex --out missingness')

# summary stats on allele frequency
get_ipython().system('plink2 --bfile dataset_15055355_plink --freq --allow-no-sex --out frequencies')

# case/control missingness
get_ipython().system('plink2 --bfile dataset_15055355_plink --test-missing --allow-no-sex --out case_con')

# haplotype
get_ipython().system('plink2 --bfile dataset_15055355_plink --test-mishap --allow-no-sex --out haplo')

# hardy-weinberg
get_ipython().system('plink2 --bfile dataset_15055355_plink --hardy --allow-no-sex --out hardy')

# heterozygozity
get_ipython().system('plink2 --bfile dataset_15055355_plink --het --allow-no-sex --out het')


### END PLINK DESCRIPTIVE ANALYSIS ###

### BEGIN QC ###

## format variants
get_ipython().system('plink2 --bfile dataset_15055355_plink --set-all-var-ids @:#:\\$r:\\$a --new-id-max-allele-len 1096 --make-bed --out dataset_15055355_temp1')

## run qc checks
# sample missingness
get_ipython().system('plink2 --bfile dataset_15055355_temp1 --remove failed_samples_to_remove.txt --make-bed --out dataset_15055355_plink_filtered')

# variant missingness
get_ipython().system('plink2 --bfile dataset_15055355_plink_filtered --exclude failed_snps_to_remove.txt --make-bed --out dataset_15055355_plink_filtered_3')

# MAF
get_ipython().system('plink2 --bfile dataset_15055355_plink_filtered_3 --maf 0.01 --make-bed --out dataset_15055355_plink_filtered_4')

# hwe
get_ipython().system('plink2 --bfile dataset_15055355_plink_filtered_4 --hwe 1e-6 --make-bed --out dataset_15055355_plink_filtered_5')

# heterozygosity
get_ipython().system('plink2 --bfile dataset_15055355_plink_filtered_5 --het --out heterozygosity')

# relatedness
get_ipython().system('plink2 --bfile dataset_15055355_plink_filtered_5 --make-king-table --out relatedness_check')

# pruning
get_ipython().system('plink2 --bfile dataset_15055355_plink_final --indep-pairwise 50 10 0.1 --out pca_pruning')


# principal components
get_ipython().system('plink2 --bfile dataset_15055355_plink_final --extract pca_pruning.prune.in --pca 10 --out my_pca_results')

### END QC ###

### BEGIN COVARIATE MERGE ###
df_pheno = pd.read_csv("covariates_with_race.txt", sep='\t', dtype={'FID': str, 'IID': str})

# load pca
df_pca = pd.read_csv("my_pca_results.eigenvec", sep='\s+', dtype={'#FID': str, 'IID': str})
df_pca.rename(columns={'#FID': 'FID'}, inplace=True)

# merge & save
data = pd.merge(df_pheno, df_pca, on=['FID', 'IID'])
data.to_csv('all_covariates.txt', sep='\t', index=False, na_rep='NA')

### END COVARIATE MERGE ###

### BEGIN GWAS PREP ###

## train/validation split
print("="*50)
print("Creating training/validation split...")
print("="*50)

# load final covariates file
df = pd.read_csv("all_covariates.txt", sep='\t', dtype={'FID': str, 'IID': str})

# split 70/30, stratifying by case control (1 = control, 2 = case)
train_df, test_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df['pain'])

print(f"Total Cohort: {len(df)}")
print(f"Training Set (70%): {len(train_df)} (Cases: {len(train_df[train_df['pain']==2])}, Controls: {len(train_df[train_df['pain']==1])})")
print(f"Validation Set (30%): {len(test_df)} (Cases: {len(test_df[test_df['pain']==2])}, Controls: {len(test_df[test_df['pain']==1])})")

# save IDs to text files
train_df[['FID', 'IID']].to_csv("train_ids1.txt", sep='\t', index=False, header=False)
test_df[['FID', 'IID']].to_csv("test_ids1.txt", sep='\t', index=False, header=False)

print("\n IDs exported")

### END GWAS PREP ###

## BEGIN GWAS ##

get_ipython().system('plink2 --bfile dataset_15055355_plink_final --keep train_ids1.txt --glm hide-covar --pheno all_covariates.txt --pheno-name pain --covar all_covariates.txt --covar-name age,bri,PC1-PC10,race_American_Indian_or_Alaska_Native,race_Asian,race_Black_or_African_American,race_I_prefer_not_to_answer,race_Middle_Eastern_or_North_African,race_More_than_one_population,race_None_Indicated,race_None_of_these,race_PMI:_Skip --covar-variance-standardize --out gwas_results')

## END GWAS ##

### BEGIN VARIANT EXTRACTION ###

# load results
df = pd.read_csv("gwas_results.pain.glm.logistic.hybrid", sep='\s+')

# filter for suggestive significance (P < 1e-5)
suggestive_snps = df[df['P'] < 1e-5]

# extract
suggestive_snps['New_ID'] = (
    suggestive_snps['#CHROM'].astype(str) + ":" +
    suggestive_snps['POS'].astype(str) + ":" +
    suggestive_snps['REF'] + ":" +
    suggestive_snps['ALT']
)

# save IDs to text file
suggestive_snps['New_ID'].to_csv("suggestive_snps.txt", index=False, header=False)

print(f"Found {len(suggestive_snps)} suggestive variants.")
print("Saved variants to suggestive_snps.txt")


# recode
get_ipython().system('plink2 --bfile dataset_15055355_plink_final --extract suggestive_snps.txt --recode A --out suggestive_genotypes')

### END VARIANT EXTRACTION ###

### BEGIN PREDICTION ANALYSIS PREP ###
print("="*50)
print("ROC/AUC data preparation, imputation, and scaling...")
print("="*50)

# load data
df_pheno = pd.read_csv("all_covariates.txt", sep='\t', dtype={'FID': str, 'IID': str})
df_geno = pd.read_csv("suggestive_genotypes.raw", sep='\s+', dtype={'FID': str, 'IID': str})
data = pd.merge(df_pheno, df_geno, on=['FID', 'IID'])

# format (1 -> 0, 2 -> 1)
data = data[data['pain'].isin([1, 2])].copy()
data['pain'] = data['pain'] - 1

# separate into train and test sets
train_ids = pd.read_csv("train_ids.txt", sep='\t', names=['FID', 'IID'], dtype=str)
test_ids = pd.read_csv("test_ids.txt", sep='\t', names=['FID', 'IID'], dtype=str)

train_data = pd.merge(train_ids, data, on=['FID', 'IID'])
test_data = pd.merge(test_ids, data, on=['FID', 'IID'])

# define variables
clinical_vars = ['age', 'bri']
candidate_snps = [col for col in df_geno.columns if col not in ['FID', 'IID', 'PAT', 'MAT', 'SEX', 'PHENOTYPE']]
pcs = ['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10']
all_vars = clinical_vars + pcs + candidate_snps

# drop patients ONLY if missing clinical variables, PCs, or phenotype
strict_complete_vars = clinical_vars + pcs + ['pain']
train_data = train_data.dropna(subset=strict_complete_vars)
test_data = test_data.dropna(subset=strict_complete_vars)

# impute missing genotypes with median from training set to prevent data leakage
for snp in candidate_snps:
    snp_median = train_data[snp].median()

    # fill missing values in both sets with training median
    train_data[snp] = train_data[snp].fillna(snp_median)
    test_data[snp] = test_data[snp].fillna(snp_median)

X_train, y_train = train_data[all_vars], train_data['pain']
X_test, y_test = test_data[all_vars], test_data['pain']

print(f"Training Set Size: {len(X_train)}")
print(f"Validation Set Size: {len(X_test)}")

# standardize data (Fit on train, transform on both to prevent data leakage)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

### END PREDICTION ANALYSIS PREP ###

### BEGIN LASSO ###

print("\n" + "="*50)
print("LASSO SELECTION & LOGISTIC REGRESSION (TRAIN SET ONLY)")
print("="*50)

# run LASSO strictly on training set
lasso_model = LogisticRegressionCV(Cs=10, cv=5, penalty='l1', solver='liblinear', random_state=42)
lasso_model.fit(X_train_scaled, y_train)

lasso_coefs = pd.Series(lasso_model.coef_[0], index=X_train.columns)
surviving_vars = [var for var, coef in lasso_coefs.items() if coef != 0.0]
print(f"LASSO retained {len(surviving_vars)} variables.")

# collinearity purge on survivors
X_train_survivors = X_train_scaled[surviving_vars].copy()
X_train_survivors.insert(0, 'const', 1.0)

independent_vars = []
current_rank = 0
for var in X_train_survivors.columns:
    test_cols = independent_vars + [var]
    new_rank = np.linalg.matrix_rank(X_train_survivors[test_cols].values)
    if new_rank > current_rank:
        independent_vars.append(var)
        current_rank = new_rank

independent_vars.remove('const')
print(f"Rank Check: {len(surviving_vars) - len(independent_vars)} perfectly collinear variables dropped.")

# fit final models on training set
X_train_final = sm.add_constant(X_train_scaled[independent_vars])
final_hybrid_model = sm.Logit(y_train, X_train_final).fit(disp=0)

valid_clinical = [v for v in clinical_vars if v in independent_vars]
valid_genetic = [v for v in (candidate_snps + pcs) if v in independent_vars]

clinical_model = sm.Logit(y_train, sm.add_constant(X_train_scaled[valid_clinical])).fit(disp=0)
genetic_model = sm.Logit(y_train, sm.add_constant(X_train_scaled[valid_genetic])).fit(disp=0)

import pandas as pd
import numpy as np

print("\n" + "="*65)
print("PHASE 2.5: HYBRID MODEL PREDICTORS (OR, 95% CI, P-VALUE)")
print("="*65)

# extract log-odds (params), 95% CI, and p-values
params = final_hybrid_model.params
conf = final_hybrid_model.conf_int()
conf.columns = ['Lower_CI', 'Upper_CI']
pvalues = final_hybrid_model.pvalues

# combine into single df and exponentiate to get ORs
results_df = pd.DataFrame({
    'Log_Odds': params,
    'Odds_Ratio': np.exp(params),
    'Lower_95_CI': np.exp(conf['Lower_CI']),
    'Upper_95_CI': np.exp(conf['Upper_CI']),
    'P_Value': pvalues
})

# drop intercept/constant
if 'const' in results_df.index:
    results_df = results_df.drop('const')

# sort by ascending p-value
results_df = results_df.sort_values('P_Value')

# print results table
print(f"{'Variable Name':<20} | {'Odds Ratio (95% CI)':<25} | {'P-Value'}")
print("-" * 65)

for var, row in results_df.iterrows():
    or_ci_str = f"{row['Odds_Ratio']:.3f} ({row['Lower_95_CI']:.3f} - {row['Upper_95_CI']:.3f})"

    p_val_str = "<0.001" if row['P_Value'] < 0.001 else f"{row['P_Value']:.4f}"

    print(f"{var:<20} | {or_ci_str:<25} | {p_val_str}")

print("="*65)

# export lasso-retained predictors to csv
csv_filename = "Hybrid_Model_Predictors_Table.csv"
results_df.to_csv(csv_filename)
print(f"\n Printed to '{csv_filename}'")

### END LASSO ###


### BEGIN ROC/AUC ###
print("\n" + "="*50)
print("Running ROC/AUC on test/validation set...")
print("="*50)

# predict on TEST set
hybrid_probs = final_hybrid_model.predict(sm.add_constant(X_test_scaled[independent_vars]))
clinical_probs = clinical_model.predict(sm.add_constant(X_test_scaled[valid_clinical]))
genetic_probs = genetic_model.predict(sm.add_constant(X_test_scaled[valid_genetic]))

# calculate AUCs
fpr_hyb, tpr_hyb, _ = roc_curve(y_test, hybrid_probs)
auc_hyb = auc(fpr_hyb, tpr_hyb)

fpr_clin, tpr_clin, _ = roc_curve(y_test, clinical_probs)
auc_clin = auc(fpr_clin, tpr_clin)

fpr_gen, tpr_gen, _ = roc_curve(y_test, genetic_probs)
auc_gen = auc(fpr_gen, tpr_gen)

# plot figure
plt.figure(figsize=(9, 7), dpi=300)
plt.plot(fpr_clin, tpr_clin, color='#1f77b4', linestyle='--', lw=2, label=f'Clinical-Only Model (AUC = {auc_clin:.3f})')
plt.plot(fpr_gen, tpr_gen, color='#2ca02c', linestyle=':', lw=2.5, label=f'Genetic-Only Model (AUC = {auc_gen:.3f})')
plt.plot(fpr_hyb, tpr_hyb, color='#d62728', linestyle='-', lw=2.5, label=f'Hybrid Model (AUC = {auc_hyb:.3f})')
plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle=':')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12, fontweight='bold')
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12, fontweight='bold')
plt.title('Predictive Performance for AIMSS Toxicity (Validation Set)', fontsize=14, fontweight='bold', pad=15)
plt.legend(loc="lower right", fontsize=11, frameon=True, shadow=True)
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("AIMSS_3_Model_ROC_Curve_Validation.png")
plt.show()

print(f"Validation Clinical-Only AUC: {auc_clin:.3f}")
print(f"Validation Genetic-Only AUC:  {auc_gen:.3f}")
print(f"Validation Hybrid Model AUC:  {auc_hyb:.3f}")

from sklearn.utils import resample
import numpy as np
from sklearn.metrics import roc_auc_score

print("\n" + "="*50)
print("Calculating 95% CI for AUCs...")
print("="*50)

n_bootstraps = 1000
boot_clin, boot_gen, boot_hyb = [], [], []

y_true = np.array(y_test)
p_clin = np.array(clinical_probs)
p_gen = np.array(genetic_probs)
p_hyb = np.array(hybrid_probs)

np.random.seed(42)

for i in range(n_bootstraps):
    indices = resample(range(len(y_true)), random_state=i)

    if len(np.unique(y_true[indices])) < 2:
        continue

    boot_clin.append(roc_auc_score(y_true[indices], p_clin[indices]))
    boot_gen.append(roc_auc_score(y_true[indices], p_gen[indices]))
    boot_hyb.append(roc_auc_score(y_true[indices], p_hyb[indices]))

def get_ci(data):
    return np.percentile(data, 2.5), np.percentile(data, 97.5)

ci_clin = get_ci(boot_clin)
ci_gen = get_ci(boot_gen)
ci_hyb = get_ci(boot_hyb)

print(f"Clinical-Only AUC: {auc_clin:.3f} (95% CI: {ci_clin[0]:.3f} - {ci_clin[1]:.3f})")
print(f"Genetic-Only AUC:  {auc_gen:.3f} (95% CI: {ci_gen[0]:.3f} - {ci_gen[1]:.3f})")
print(f"Hybrid Model AUC:  {auc_hyb:.3f} (95% CI: {ci_hyb[0]:.3f} - {ci_hyb[1]:.3f})")
print("="*50)

### END ROC/AUC ###

### BEGIN RISK STRATIFICATION ###

print("\n" + "="*50)
print("Running risk stratification on test/validation set...")
print("="*50)

# recalculate ROC curve to get specific thresholds from test set
fpr_hybrid, tpr_hybrid, thresholds = roc_curve(y_test, hybrid_probs)

# calculate Youden's J statistic
j_scores = tpr_hybrid - fpr_hybrid
optimal_idx = np.argmax(j_scores)
optimal_threshold = thresholds[optimal_idx]

print("1. YOUDEN'S J STATISTIC & CONFUSION MATRIX")
print(f"Optimal Cutoff Threshold (Maximized Sens/Spec): {optimal_threshold:.3f}\n")

# convert continuous probabilities into binary predictions (1 = high risk, 0 = low risk)
hybrid_binary_preds = (hybrid_probs >= optimal_threshold).astype(int)

# generate confusion matrix
tn, fp, fn, tp = confusion_matrix(y_test, hybrid_binary_preds).ravel()

# calculate metrics
ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
npv = tn / (tn + fn) if (tn + fn) > 0 else 0
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print(f"True Positives (Correctly flagged High Risk): {tp}")
print(f"True Negatives (Correctly flagged Low Risk):  {tn}")
print(f"False Positives (False Alarms):               {fp}")
print(f"False Negatives (Missed Diagnoses):           {fn}")
print("-" * 30)
print(f"Positive Predictive Value (PPV): {ppv:.3f} (If flagged, {ppv*100:.1f}% got AIMSS)")
print(f"Negative Predictive Value (NPV): {npv:.3f} (If cleared, {npv*100:.1f}% stayed pain-free)")
print(f"Sensitivity (Recall):            {sensitivity:.3f}")
print(f"Specificity:                     {specificity:.3f}")

### END RISK STRATIFICATION ###

### BEGIN QUINTILE ANALYSIS ###
print("\n" + "="*50)
print("Running quintile analysis on test/validation set...")
print("="*50)

# create df strictly using test/validation outcome and test/validation probabilities
risk_df = pd.DataFrame({
    'True_Outcome': y_test.values,
    'Predicted_Prob': hybrid_probs
})

# slice cohort into 5 quintiles based on score
try:
    risk_df['Quintile'] = pd.qcut(risk_df['Predicted_Prob'], 5, labels=['Q1 (Bottom 20%)', 'Q2', 'Q3', 'Q4', 'Q5 (Top 20%)'], duplicates='drop')

    # calculate actual AIMSS prevalence in each quintile
    quintile_stats = risk_df.groupby('Quintile', observed=False)['True_Outcome'].agg(['count', 'mean']).rename(columns={'count': 'Total Patients', 'mean': 'Actual AIMSS Prevalence'})

    print("\nDisease Prevalence by Risk Tier:")
    print(quintile_stats.to_string(formatters={'Actual AIMSS Prevalence': '{:.1%}'.format}))

    # calculate OR between top 20% and bottom 20%
    q5_cases = risk_df[(risk_df['Quintile'] == 'Q5 (Top 20%)') & (risk_df['True_Outcome'] == 1)].shape[0]
    q5_controls = risk_df[(risk_df['Quintile'] == 'Q5 (Top 20%)') & (risk_df['True_Outcome'] == 0)].shape[0]

    q1_cases = risk_df[(risk_df['Quintile'] == 'Q1 (Bottom 20%)') & (risk_df['True_Outcome'] == 1)].shape[0]
    q1_controls = risk_df[(risk_df['Quintile'] == 'Q1 (Bottom 20%)') & (risk_df['True_Outcome'] == 0)].shape[0]

    # ORs
    odds_q5 = q5_cases / q5_controls if q5_controls > 0 else np.nan
    odds_q1 = q1_cases / q1_controls if q1_controls > 0 else np.nan

    if odds_q1 > 0:
        stratification_or = odds_q5 / odds_q1
        print("\nCLINICAL IMPACT STATEMENT:")
        print(f"Patients in the Top 20% of the Hybrid Model risk scores are {stratification_or:.2f} times")
        print("more likely to develop AIMSS than those in the Bottom 20%.")
    else:
        print("\n(Odds Ratio cannot be calculated because Q1 had zero cases.)")

except ValueError as e:
    print(f"\nCould not calculate exact quintiles due to score clustering: {e}")

print("="*50)

### END QUINTILE ANALYSIS ###

### BEGIN PC, QQ & MANHATTAN PLOTS ###

## principal component (PC) plots
EIGENVEC_FILE = "my_pca_results.eigenvec"

print(f"Loading eigenvector file: {EIGENVEC_FILE}")

try:
    df_pca = pd.read_csv(EIGENVEC_FILE, sep='\t')

    df_pca.rename(columns={'#FID': 'FID'}, inplace=True)

except FileNotFoundError:
    print(f"Error: Could not find {EIGENVEC_FILE}")
    print("Please make sure Step 2 ran correctly.")
    exit()
except Exception as e:
    print(f"Error reading file: {e}")
    print("Is the file empty?")
    exit()

print(f"File loaded. Plotting PC1 vs PC2...")

# check if PC columns exist
if 'PC1' not in df_pca.columns or 'PC2' not in df_pca.columns:
    print("Error: PC1 or PC2 not found in file. Header is:")
    print(df_pca.columns)
    exit()

# create plot
plt.figure(figsize=(10, 8))
plt.scatter(df_pca['PC1'], df_pca['PC2'], alpha=0.5)
plt.title('PCA of Genetic Ancestry', fontsize=16)
plt.xlabel('Principal Component 1', fontsize=12)
plt.ylabel('Principal Component 2', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

# plot PC2 vs PC3
if 'PC3' in df_pca.columns:
    plt.figure(figsize=(10, 8))
    plt.scatter(df_pca['PC2'], df_pca['PC3'], alpha=0.5, color='green')
    plt.title('PCA of Genetic Ancestry', fontsize=16)
    plt.xlabel('Principal Component 2', fontsize=12)
    plt.ylabel('Principal Component 3', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.show()

# plot PC3 vs PC4
if 'PC3' in df_pca.columns:
    plt.figure(figsize=(10, 8))
    plt.scatter(df_pca['PC3'], df_pca['PC4'], alpha=0.5, color='green')
    plt.title('PCA of Genetic Ancestry', fontsize=16)
    plt.xlabel('Principal Component 3', fontsize=12)
    plt.ylabel('Principal Component 4', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.show()

# plot PC4 vs PC5
if 'PC4' in df_pca.columns:
    plt.figure(figsize=(10, 8))
    plt.scatter(df_pca['PC4'], df_pca['PC5'], alpha=0.5, color='green')
    plt.title('PCA of Genetic Ancestry', fontsize=16)
    plt.xlabel('Principal Component 4', fontsize=12)
    plt.ylabel('Principal Component 5', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.show()

# plot PC5 vs PC6
if 'PC5' in df_pca.columns:
    plt.figure(figsize=(10, 8))
    plt.scatter(df_pca['PC5'], df_pca['PC6'], alpha=0.5, color='green')
    plt.title('PCA of Genetic Ancestry', fontsize=16)
    plt.xlabel('Principal Component 5', fontsize=12)
    plt.ylabel('Principal Component 6', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.show()

# plot PC6 vs PC7
if 'PC6' in df_pca.columns:
    plt.figure(figsize=(10, 8))
    plt.scatter(df_pca['PC6'], df_pca['PC7'], alpha=0.5, color='green')
    plt.title('PCA of Genetic Ancestry', fontsize=16)
    plt.xlabel('Principal Component 5', fontsize=12)
    plt.ylabel('Principal Component 6', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.show()

# plot PC7 vs PC8
if 'PC7' in df_pca.columns:
    plt.figure(figsize=(10, 8))
    plt.scatter(df_pca['PC7'], df_pca['PC8'], alpha=0.5, color='green')
    plt.title('PCA of Genetic Ancestry', fontsize=16)
    plt.xlabel('Principal Component 5', fontsize=12)
    plt.ylabel('Principal Component 6', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.show()

# plot PC8 vs PC9
if 'PC8' in df_pca.columns:
    plt.figure(figsize=(10, 8))
    plt.scatter(df_pca['PC8'], df_pca['PC9'], alpha=0.5, color='green')
    plt.title('PCA of Genetic Ancestry', fontsize=16)
    plt.xlabel('Principal Component 5', fontsize=12)
    plt.ylabel('Principal Component 6', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.show()

# plot PC9 vs PC10
if 'PC9' in df_pca.columns:
    plt.figure(figsize=(10, 8))
    plt.scatter(df_pca['PC9'], df_pca['PC10'], alpha=0.5, color='green')
    plt.title('PCA of Genetic Ancestry', fontsize=16)
    plt.xlabel('Principal Component 5', fontsize=12)
    plt.ylabel('Principal Component 6', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.show()


## qq plot
gwas_file = "gwas_results3.pain.glm.logistic.hybrid"

SIG_THRESHOLD = 1e-5 # for manhattan plot, keep all SNPs with P < this
NON_SIG_SAMPLE_RATE = 0.001 # (0.1%)

try:
    with open(gwas_file, 'r') as f:
        pass
except FileNotFoundError:
    print(f"\n*** Error: Could not find the results file: {gwas_file}")
    print("Please check the 'gwas_file' variable in the script.")
    sys.exit()

print(f"\n--- Part 1: Generating Q-Q Plot ---")
print(f"Loading *only* the P-value column (memory-safe)...")

try:
    p_values = pd.read_csv(
        gwas_file,
        sep='\t',
        usecols=['P'],
        na_values=['NA', '.'],
        dtype={'P': np.float64}
    ).dropna()

    if p_values.empty:
        print("Error: No p-values loaded. Is 'P' column correct?")
        sys.exit()

    print(f"Loaded {len(p_values)} p-values.")

    # create plot
    print("Creating plot...")
    fig = sm.qqplot(p_values['P'], line='s', markersize=2)
    plt.title("Q-Q Plot of Association Results")
    plt.xlabel("Expected -log10(P)")
    plt.ylabel("Observed -log10(P)")
    plt.savefig("gwas_qq_plot.png")
    plt.show()

    print("Q-Q Plot saved as gwas_qq_plot.png")

except Exception as e:
    print(f"\n*** Error during Q-Q plot creation: {e}")
    print("Please check the 'P' column name and file format.")


# manhattan plot
print(f"\n--- Part 2: Generating Manhattan Plot (Subsampling) ---")
print("Reading file in chunks to save memory (this may take a minute)...")

try:
    chunk_iter = pd.read_csv(
        "gwas_results3.pain.glm.logistic.hybrid",
        sep='\t',
        usecols=['#CHROM', 'POS', 'P'],
        na_values=['NA', '.'],
        chunksize=1000000
    )

    light_dfs = []
    for chunk in chunk_iter:
        chunk.dropna(inplace=True)

        # keep all SNPs significant at aforedefined threshold
        sig_chunk = chunk[chunk['P'] < SIG_THRESHOLD]

        non_sig_chunk = chunk[chunk['P'] >= SIG_THRESHOLD]
        non_sig_sampled = non_sig_chunk.sample(frac=NON_SIG_SAMPLE_RATE, random_state=1)

        light_dfs.append(pd.concat([sig_chunk, non_sig_sampled]))

    print("Assembling final plot data...")

    df_plot = pd.concat(light_dfs)

    def chrom_to_num(c):
        c = str(c).upper().replace('CHR', '')
        if c == 'X': return 23
        if c == 'Y': return 24
        if c == 'MT' or c == 'M': return 25
        try: return int(c)
        except: return 99

    df_plot['chrom_num'] = df_plot['#CHROM'].apply(chrom_to_num)

    # sort numerically by new column, then by position
    df_plot = df_plot.sort_values(['chrom_num', 'POS'])

    # create -log10(P) column safely
    df_plot['-log10p'] = -np.log10(df_plot['P'])

    # calculate cumulative positions for x-axis and exact midpoints for labels
    df_plot['cum_pos'] = 0
    s = 0
    x_labels = []
    x_ticks = []

    for chrom_num, group in df_plot.groupby('chrom_num', sort=False):
        chrom_length = group['POS'].max()

        df_plot.loc[df_plot['chrom_num'] == chrom_num, 'cum_pos'] = group['POS'] + s

        mid_point = s + (chrom_length / 2)
        x_ticks.append(mid_point)
        x_labels.append(group['#CHROM'].iloc[0]) # Get original name (e.g., '1', 'X')

        s += chrom_length

    print("Creating plot...")
    plt.figure(figsize=(16, 6), dpi=300)

    # alternate dark blue and grey
    colors = ['#1f77b4', '#7f7f7f']

    for i, (chrom_num, group) in enumerate(df_plot.groupby('chrom_num', sort=False)):
        plt.scatter(
            group['cum_pos'],
            group['-log10p'],
            color=colors[i % len(colors)],
            s=6,
            alpha=0.8
        )

    # add genome-wide significance line
    plt.axhline(y=-np.log10(5e-8), color='red', linestyle='--', linewidth=1.2)

    # format axes
    plt.title("Manhattan Plot of AIMSS Toxicity Genotype-Phenotype Associations", fontsize=15, fontweight='bold', pad=15)
    plt.ylabel("-log10(P-value)", fontsize=13, fontweight='bold')
    plt.xlabel("Chromosome", fontsize=13, fontweight='bold')

    # add chrom names
    plt.xticks(x_ticks, x_labels, fontsize=10)

    # final clean up
    plt.xlim([0, s])
    plt.ylim(bottom=0)

    plt.tight_layout()
    plt.savefig("gwas_manhattan_plot_final1.png")
    plt.show()

    print("Manhattan Plot formatted and saved as gwas_manhattan_plot_final.png")

except Exception as e:
    print(f"\n*** Error during Manhattan plot creation: {e}")
### END PC, QQ & MANHATTAN PLOTS ###

### BEGIN RSID RETRIEVAL ###
print("\n" + "="*50)
print("Get rsIDs")
print("="*50)

# load gwas results
file_path = "gwas_results3.pain.glm.logistic.hybrid"

print(f"Loading {file_path}... (This might take a minute or two)")

gwas_data = pd.read_csv(file_path, sep='\s+', low_memory=False)

# clean up column header
if '#CHROM' in gwas_data.columns:
    gwas_data.rename(columns={'#CHROM': 'CHROM'}, inplace=True)

# ensure numeric p
gwas_data['P'] = pd.to_numeric(gwas_data['P'], errors='coerce')

# filter
if 'TEST' in gwas_data.columns:
    gwas_data = gwas_data[gwas_data['TEST'] == 'ADD']

# apply suggestive threshold
suggestive_threshold = 1e-5
suggestive_snps = gwas_data[gwas_data['P'] < suggestive_threshold].copy()

print(f"\n Filtered down to {len(suggestive_snps)} suggestive variants (P < {suggestive_threshold}).")

# ensembl api lookup
def get_rsid_from_ensembl(chrom, pos):
    """Hits the Ensembl REST API to find the rsID for a specific GRCh38 coordinate."""
    server = "https://rest.ensembl.org"
    ext = f"/overlap/region/human/{chrom}:{pos}-{pos}?feature=variation"

    try:
        r = requests.get(server + ext, headers={"Content-Type": "application/json"})
        if not r.ok:
            return "API_Error"

        data = r.json()
        if data:
            rsids = [var['id'] for var in data if var['id'].startswith('rs')]
            if rsids:
                return rsids[0]
        return "Novel/No_rsID"
    except Exception as e:
        return "Connection_Error"

# get rsIDs
print("Fetching official GRCh38 rsIDs from Ensembl...\n")

results_list = []
for index, row in suggestive_snps.iterrows():
    chrom = str(row['CHROM']).replace('chr', '')
    pos = row['POS']

    risk_allele = row['A1']
    p_val = row['P']

    # based ORs
    if 'OR' in row.index:
        odds_ratio = row['OR']
        beta = np.log(odds_ratio)
    elif 'BETA' in row.index:
        beta = row['BETA']
        odds_ratio = np.exp(beta)
    else:
        odds_ratio = np.nan
        beta = np.nan

    # edge case checks
    if 'L95' in row.index and 'U95' in row.index:
        if 'OR' in row.index:
            lower_ci = row['L95']
            upper_ci = row['U95']
        else:
            lower_ci = np.exp(row['L95'])
            upper_ci = np.exp(row['U95'])

    elif 'SE' in row.index or 'LOG(OR)_SE' in row.index:
        se_col = 'SE' if 'SE' in row.index else 'LOG(OR)_SE'
        se = row[se_col]

        lower_ci = np.exp(beta - (1.96 * se))
        upper_ci = np.exp(beta + (1.96 * se))

    else:
        lower_ci = np.nan
        upper_ci = np.nan

    print(f"Querying Chr {chrom} at position {pos}...")

    rsid = get_rsid_from_ensembl(chrom, pos)

    results_list.append({
        'Chromosome': chrom,
        'Position': pos,
        'Risk_Allele': risk_allele,
        'Odds_Ratio': odds_ratio,
        'Lower_95_CI': lower_ci,
        'Upper_95_CI': upper_ci,
        'P_Value': p_val,
        'GRCh38_rsID': rsid
    })

    # sleep
    time.sleep(0.1)

# export table
final_df = pd.DataFrame(results_list)

print("\n" + "="*50)
print("Results preview:")
print("="*50)
print(final_df.head(10).to_string(index=False))

output_filename = "Suggestive_Variants_Filtered_Annotated2.csv"
final_df.to_csv(output_filename, index=False)
print(f"\n Results saved to '{output_filename}'")

### END RSID RETRIEVAL ###

### BEGIN CANDIDATE VARIANT RETRIEVAL ###

print("\n" + "="*50)
print("Retrieving candidate variants...")
print("="*50)

# candidate variants
target_rsids = [
    'rs6579255', 'rs6060709', 'rs9525016', 'rs8050136', 'rs3740387',
    'rs7198606', 'rs9939609', 'rs4468641', 'rs7949972', 'rs2303223',
    'rs749671', 'rs5758651', 'rs17713054', 'rs10760115', 'rs203878',
    'rs8052655', 'rs1043620', 'rs6457150', 'rs13708', 'rs1064608',
    'rs7184253', 'rs1049073', 'rs6550024', 'rs9922085', 'rs8044843'
]

# get GRCh38 coords from dnsembl
def get_coords_from_rsid(rsid):
    server = "https://rest.ensembl.org"
    ext = f"/variation/human/{rsid}"
    try:
        r = requests.get(server+ext, headers={"Content-Type": "application/json"})
        if r.status_code == 200:
            data = r.json()

            for mapping in data.get('mappings', []):
                if mapping['assembly_name'] == 'GRCh38':
                    return str(mapping['seq_region_name']), int(mapping['start'])
    except Exception as e:
        pass
    return None, None

# lookup dictionary
snp_lookup = []
for rsid in target_rsids:
    chrom, pos = get_coords_from_rsid(rsid)
    if chrom and pos:
        snp_lookup.append({'rsID': rsid, 'chrom': chrom, 'pos': pos})

    # sleep
    time.sleep(0.1)

lookup_df = pd.DataFrame(snp_lookup)
print(f"Mapped {len(lookup_df)} out of {len(target_rsids)} candidate SNPs.")

### END CANDIDATE VARIANT RETRIEVAL ###

### BEGIN FINAL EXTRACTION OF ALL VARIANT STATS ###
print("\n" + "="*50)
print("Extracting all variant stats...")
print("="*50)

gwas_data = pd.read_csv("gwas_results3.pain.glm.logistic.hybrid", sep='\s+')

results_list = []

# search gwas data using coordinates
for index, target in lookup_df.iterrows():
    t_chrom = target['chrom']
    t_pos = target['pos']
    t_rsid = target['rsID']

    # filter gwas data
    match = gwas_data[
        (gwas_data['CHROM'].astype(str).str.replace('chr', '') == t_chrom) &
        (gwas_data['POS'] == t_pos)
    ]

    # if match found
    if not match.empty:
        # take first matched row
        row = match.iloc[0]

        risk_allele = row.get('A1', np.nan)
        p_val = row.get('P', np.nan)

        # OR & 95% CI
        if 'OR' in row:
            odds_ratio = row['OR']
            beta = np.log(odds_ratio) if odds_ratio > 0 else np.nan
        elif 'BETA' in row:
            beta = row['BETA']
            odds_ratio = np.exp(beta)
        else:
            odds_ratio, beta = np.nan, np.nan

        if 'L95' in row and 'U95' in row:
            lower_ci = row['L95'] if 'OR' in row else np.exp(row['L95'])
            upper_ci = row['U95'] if 'OR' in row else np.exp(row['U95'])
        elif 'SE' in row or 'LOG(OR)_SE' in row:
            se_col = 'SE' if 'SE' in row else 'LOG(OR)_SE'
            se = row[se_col]
            lower_ci = np.exp(beta - (1.96 * se))
            upper_ci = np.exp(beta + (1.96 * se))
        else:
            lower_ci, upper_ci = np.nan, np.nan

        results_list.append({
            'rsID': t_rsid,
            'Chromosome': t_chrom,
            'Position': t_pos,
            'Risk_Allele': risk_allele,
            'Odds_Ratio': odds_ratio,
            'Lower_95_CI': lower_ci,
            'Upper_95_CI': upper_ci,
            'P_Value': p_val
        })
    else:
        # clarify if variant failed QC
        results_list.append({
            'rsID': t_rsid,
            'Chromosome': t_chrom,
            'Position': t_pos,
            'Risk_Allele': "Not Found in GWAS",
            'Odds_Ratio': np.nan,
            'Lower_95_CI': np.nan,
            'Upper_95_CI': np.nan,
            'P_Value': np.nan
        })

# final df
candidate_results_df = pd.DataFrame(results_list)

# print table
pd.set_option('display.max_rows', None)
display(candidate_results_df)
pd.reset_option('display.max_rows')

# export to csv
candidate_results_df.to_csv("Candidate_SNPs_Results.csv", index=False)
print("\n Results exported to 'Candidate_SNPs_Results.csv'")

### END FINAL EXTRACTION OF ALL VARIANT STATS ###

print("GENETIC ANALYSES COMPLETE!")

#### END GENETIC ANALYSES ####